# GISLR — ASL gloss → English: reverse rules vs Workers AI

**What this notebook does.** Pipeline stage 3 of sign → speech (TODO §12.6): turn the
accepted gloss sequence (ASL order, no inflection) into an English sentence for TTS. It
compares:

| engine | where it would run | what it is |
|---|---|---|
| `identity` | anywhere | the glosses joined, capitalized. The **lower bound** |
| `rules_v1` | client or edge, free | `sb.rescore.gloss2en`: the reverse of `sb.synthesize.gloss.rules_v2`. It restores articles, the copula, tense, `do`-support and pronoun case, driven by the corpus lexicon's POS tags |
| `llm:<model>` | edge (Workers AI, Free plan) | `prompts/v1/gloss2en.txt` (hashed into every result), temperature 0 |

**Two scores, because references are scarce:**
1. **References.** `sb.rescore.evalset/gloss2en.v1.jsonl`: 132 sentences (12 per
   theme, seeded), 1–2 English references each. BLEU and chrF (sacrebleu, multi-ref).
   **The references are Claude-written drafts awaiting user review**, so treat them as
   development numbers.
2. **Round trip, no references needed.** English → `rules_v2` → GISLR labels
   (`gloss2en.to_gislr`), compared with the original glosses. The measures are **content
   recall** (share of the original glosses that come back) and gloss WER. This checks
   that the meaning survived. It says nothing about fluency, so a fluent sentence that
   `rules_v2` parses differently can lose points.

**Samples:** the eval set, plus `roundtrip_sample` (300) seeded corpus sentences for
the LLMs. They exclude the eval set and the prompt's few-shot examples. The rules
engine runs on every corpus sentence (1,757).

**Artifacts** (`data/cache/gislr/downstream/gloss2en/`): `outputs_<engine>.jsonl`, the
LLM response cache `llm_cache_<model>.jsonl` (resumable, keyed by model + prompt hash +
input), `results.json`, `results_table.csv`.

**Credentials:** the LLM section needs `CLOUDFLARE_ACCOUNT_ID` and
`CLOUDFLARE_API_TOKEN` (Workers AI: Read) in the environment or the repo `.env`.
Without them it prints how to set them and skips. Cost on the Free plan: about 430 calls
per model, well under the 10,000-neuron daily allowance.

## Setup

In [1]:
# ============================================================
# Imports, config, corpus, eval set, samples
# ============================================================
import json
import os
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import sacrebleu
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR
from sb.recognize.sequences.corpus import load_lexicon, load_sentences
from sb.rescore import client as C
from sb.rescore import gloss2en as G
from sb.synthesize.gloss import rules_v2
from sb.synthesize.metrics import edit_ops

CONFIG = json.loads((EXPERIMENTS_DIR / "recognition" / "configs" / "gislr.gloss2en.json").read_text(encoding="utf-8"))

OUT = CACHE_DIR / "gislr" / "downstream" / "gloss2en"
OUT.mkdir(parents=True, exist_ok=True)
EVALSET_PATH = Path(C.__file__).parent / "evalset" / f"{CONFIG['evalset']}.jsonl"


def write_json(path: Path, obj) -> None:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=float), encoding="utf-8")
    os.replace(tmp, path)


def write_jsonl(path: Path, rows: list[dict]) -> None:
    tmp = path.with_suffix(".tmp")
    tmp.write_text("".join(json.dumps(r) + "\n" for r in rows), encoding="utf-8")
    os.replace(tmp, path)


SENTS = load_sentences(CONFIG["corpus_version"])
LEX = load_lexicon(CONFIG["corpus_version"])
VOCAB = set(LEX)
EVALSET = [json.loads(l) for l in EVALSET_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
EVAL_IDS = {r["id"] for r in EVALSET}
assert not EVAL_IDS & set(CONFIG["fewshot_ids"]), "a few-shot example is in the eval set"
rng = np.random.default_rng(CONFIG["seed"])
pool = [s for s in SENTS if s.id not in EVAL_IDS and s.id not in CONFIG["fewshot_ids"]]
SAMPLE = [pool[i] for i in sorted(rng.choice(len(pool), CONFIG["roundtrip_sample"], replace=False))]
PROMPT, PROMPT_SHA = C.load_prompt(CONFIG["prompt_version"], "gloss2en")
print(f"corpus {len(SENTS)} · eval set {len(EVALSET)} · round-trip sample {len(SAMPLE)} · prompt sha {PROMPT_SHA[:12]}")

corpus 1757 · eval set 132 · round-trip sample 300 · prompt sha a69a1aab56a6


## 1. Engines on the whole corpus / sample

`identity` and `rules_v1` run on every corpus sentence (cheap, deterministic). Their
outputs are rewritten every run.

In [2]:
# ============================================================
# identity + rules_v1 over every corpus sentence
# ============================================================
N_SHOW = 15


def identity(glosses) -> str:
    s = " ".join(G.SURFACE.get(g, g) for g in glosses)
    return s[:1].upper() + s[1:] + "."


for name, fn in (("identity", identity), ("rules_v1", lambda g: G.convert(g, LEX))):
    write_jsonl(OUT / f"outputs_{name}.jsonl",
                [{"id": s.id, "gloss": " ".join(s.glosses), "english": fn(list(s.glosses))} for s in SENTS])

show = pd.read_json(OUT / "outputs_rules_v1.jsonl", lines=True).sample(N_SHOW, random_state=0)
show[["gloss", "english"]]

,gloss,english
1379,toothbrush tooth morning night,The toothbrush tooth in the morning at night.
668,that girl minemy aunt child,That is my girl aunt child.
412,yes minemy hungry,"Yes, I am hungry."
558,all finish,Everything finished.
1363,tongue owie hot pizza,The tongue owie is hot with pizza.
1320,toy where,Where is the toy?
1507,fish for kitty,Fish is for the kitty.
1010,open glasswindow hot,Open the window hot.
1681,now time food,Now the time food.
1756,wake mom please,"Wake up Mom, please."


## 2. Workers AI LLMs (skipped without credentials)

One call per sentence (eval set + sample), temperature 0, cached on disk. A re-run only
pays for sentences not yet cached. A prompt edit changes the hash, so old answers are
never reused.

In [3]:
# ============================================================
# LLM gloss -> English, per model, cached
# ============================================================
TARGETS = [(r["id"], r["gloss"]) for r in EVALSET] + [(s.id, " ".join(s.glosses)) for s in SAMPLE]

if C.credentials() is None:
    print("No Workers AI credentials: set CLOUDFLARE_ACCOUNT_ID and CLOUDFLARE_API_TOKEN "
          "(token permission 'Workers AI: Read') in the repo .env, then re-run this cell.")
else:
    for model in CONFIG["models"]:
        slug = re.sub(r"[^a-z0-9.]+", "-", model.lower()).strip("-")
        client = C.WorkersAI(model, OUT / f"llm_cache_{slug}.jsonl", max_tokens=CONFIG["max_tokens"])
        rows, lat = [], []
        for sid, gloss in tqdm(TARGETS, desc=model):
            r = client.chat(PROMPT, PROMPT_SHA, gloss)
            rows.append({"id": sid, "gloss": gloss, "english": C.clean_sentence(r["text"]), "raw": r["text"],
                         "model": model, "prompt_sha256": PROMPT_SHA})
            if not r["cached"]:
                lat.append(r["latency_s"])
        write_jsonl(OUT / f"outputs_llm-{slug}.jsonl", rows)
        if lat:
            print(f"{model}: {len(lat)} new calls, median latency {np.median(lat):.2f}s")

No Workers AI credentials: set CLOUDFLARE_ACCOUNT_ID and CLOUDFLARE_API_TOKEN (token permission 'Workers AI: Read') in the repo .env, then re-run this cell.


## 3. Scores: references (eval set) and round trip (eval set + sample)

- **BLEU / chrF**: sacrebleu corpus scores against every reference (13a tokenizer,
  since this is English, not gloss).
- **exact**: normalized output equals one of the references.
- **round trip**: English → `rules_v2` → `to_gislr`. Measured as `content_recall` (the
  multiset share of original glosses recovered), `extra` (recovered tokens that are not
  in the original, i.e. invented content) and `gloss_wer`.

In [4]:
# ============================================================
# Reference metrics + round-trip metrics for every engine found on disk
# ============================================================
REFS = {r["id"]: r["refs"] for r in EVALSET}
GLOSS = {s.id: list(s.glosses) for s in SENTS}
SAMPLE_IDS = {s.id for s in SAMPLE}


def norm(t: str) -> str:
    return re.sub(r"[^a-z' ]", "", t.lower()).strip()


def roundtrip(english: str, gloss: list[str]) -> dict:
    back = G.to_gislr(rules_v2.convert(english), VOCAB) if english else []
    s, d, i = edit_ops(gloss, back)
    got, want = Counter(back), Counter(gloss)
    return {"back": " ".join(back), "gloss_wer": (s + d + i) / len(gloss),
            "content_recall": sum((got & want).values()) / len(gloss),
            "extra": sum((got - want).values()) / len(gloss), "exact_back": back == gloss}


rows, examples = [], []
for path in sorted(OUT.glob("outputs_*.jsonl")):
    engine = path.stem.removeprefix("outputs_")
    out = {r["id"]: r["english"] for r in map(json.loads, path.read_text(encoding="utf-8").splitlines())}
    ev = [i for i in REFS if i in out]
    n_ref = max(len(REFS[i]) for i in ev)
    hyps = [out[i] for i in ev]
    ref_streams = [[(REFS[i] + REFS[i][:1] * n_ref)[k] for i in ev] for k in range(n_ref)]
    rt_ids = [i for i in out if i in REFS or i in SAMPLE_IDS]
    rts = [roundtrip(out[i], GLOSS[i]) for i in tqdm(rt_ids, desc=f"round trip {engine}")]
    rows.append({"engine": engine, "n_eval": len(ev), "n_roundtrip": len(rt_ids),
                 "bleu": sacrebleu.corpus_bleu(hyps, ref_streams).score,
                 "chrf": sacrebleu.corpus_chrf(hyps, ref_streams).score,
                 "exact": float(np.mean([norm(out[i]) in {norm(r) for r in REFS[i]} for i in ev])),
                 **{k: float(np.mean([r[k] for r in rts])) for k in ("content_recall", "extra", "gloss_wer", "exact_back")}})
    for i, r in zip(rt_ids, rts):
        if i in REFS:
            examples.append({"id": i, "engine": engine, "gloss": " ".join(GLOSS[i]), "english": out[i],
                             "ref": REFS[i][0], "back": r["back"], "content_recall": r["content_recall"]})
TABLE = pd.DataFrame(rows).sort_values("bleu", ascending=False)
write_json(OUT / "results.json", {"prompt_sha256": PROMPT_SHA, "results": rows, "examples": examples})
TABLE.to_csv(OUT / "results_table.csv", index=False)
TABLE.round(3)

round trip identity:   0%|          | 0/432 [00:00<?, ?it/s]

round trip rules_v1:   0%|          | 0/432 [00:00<?, ?it/s]

,engine,n_eval,n_roundtrip,bleu,chrf,exact,content_recall,extra,gloss_wer,exact_back
1,rules_v1,132,432,60.232,75.011,0.424,0.925,0.055,0.194,0.600
0,identity,132,432,7.902,46.773,0.068,0.935,0.022,0.152,0.676


## 4. Side by side on the eval set

In [5]:
# ============================================================
# Every engine's output per eval sentence (first 30), plus the reference
# ============================================================
N_SIDE = 30

ex = pd.DataFrame(json.loads((OUT / "results.json").read_text())["examples"])
side = ex.pivot_table(index=["id", "gloss", "ref"], columns="engine", values="english", aggfunc="first").reset_index()
pd.set_option("display.max_colwidth", 60)
side.head(N_SIDE)

engine,id,gloss,ref,identity,rules_v1
0,animals-013,dog look cat hide tree,The dog looks at the cat hiding in the tree.,Dog look cat hide tree.,The dog looks at the cat hide the tree.
1,animals-014,cat hide bedroom,The cat is hiding in the bedroom.,Cat hide bedroom.,The cat hides the bedroom.
2,animals-031,cat like fish,The cat likes fish.,Cat like fish.,The cat likes fish.
3,animals-064,horse like carrot,The horse likes carrots.,Horse like carrot.,The horse likes the carrot.
4,animals-065,horse like apple,The horse likes apples.,Horse like apple.,The horse likes the apple.
5,animals-082,bee on flower,The bee is on the flower.,Bee on flower.,The bee is on the flower.
6,animals-096,elephant old happy,The old elephant is happy.,Elephant old happy.,The elephant is old and happy.
7,animals-106,alligator mouth open,The alligator's mouth is open.,Alligator mouth open.,The alligator mouth opens.
8,animals-113,animal outside many,There are many animals outside.,Animal outside many.,Many animal outsides.
9,animals-129,mom not like mouse,Mom does not like mice.,Mom not like mouse.,Mom does not like the mouse.


## 5. Reading the results (fill in after the run)

For `docs/reports/downstream-gloss2en.md`:
- Does `rules_v1` beat `identity` by enough to be worth keeping as the offline/free
  path, or as the fallback when the edge is unreachable?
- Do the LLMs keep the content (`content_recall`, `extra` = invented content) while
  improving fluency (BLEU/chrF)? `extra` is the safety number: an LLM that adds words
  the user never signed is the failure to watch for.
- Which model gives the best quality per latency? Workers AI Free-plan latency is in §2.
- Where both fail: the telegraphic sentences (`cut napkin scissors`). Is a
  clarification UI needed instead of a guess?